In [ ]:
import statistics
import time

import requests

URL = "http://ollama:11434"
MODEL = "tev1:0.8b"

In [4]:
models = requests.get(URL + "/api/tags").json()["models"]
print(f"{'model':<20}{'params':>9}{'quant':>9}{'size GB':>9}  capabilities")
for m in models:
    d = m["details"]
    caps = ", ".join(m.get("capabilities", []))
    print(f"{m['name']:<20}{d['parameter_size']:>9}{d['quantization_level']:>9}{m['size'] / 1e9:>9.1f}  {caps}")

model                  params    quant  size GB  capabilities
clef-flash:latest        9.1B     Q8_0     10.9  decision
tev1:0.8b             752.39M     Q8_0      0.8  decision


In [ ]:
URL = URL + "/v1/systemone"

# Mocked Check24 verticals: key -> description the model uses as routing criteria
VERTICALS = {
    "strom_gas": "Strom- und Gastarife vergleichen, Energieanbieter wechseln",
    "kfz_versicherung": "Kfz-Versicherung fuer Auto oder Motorrad vergleichen und wechseln",
    "internet_dsl": "DSL, Kabel, Glasfaser und Internetanschluss fuer zu Hause",
    "handy": "Handytarife, Mobilfunkvertraege, Prepaid und Smartphones mit Vertrag",
    "kredit": "Ratenkredit, Autokredit, Umschuldung und Baufinanzierung",
    "konto_karte": "Girokonto, Kreditkarte, Tagesgeld und Festgeld",
    "versicherungen": "Haftpflicht, Hausrat, Rechtsschutz, Kranken- und Lebensversicherung",
    "reise": "Pauschalreisen, Hotels, Ferienwohnungen und Urlaubsangebote",
    "flug": "Flugtickets und Flugverbindungen suchen und buchen",
    "mietwagen": "Mietwagen und Transporter am Urlaubsort oder vor Ort mieten",
    "shopping": "Produkte kaufen und Preise vergleichen, z.B. Elektronik und Haushalt",
    "none": "Keine passende Vergleichskategorie: Kundenservice, Smalltalk oder unklare Anfrage",
}

# Mocked redirect targets
REDIRECT_URLS = {key: f"https://www.check24.de/{key.replace('_', '-')}/" for key in VERTICALS if key != "none"}

# Redirect only above this confidence, otherwise stay on the search page
CONFIDENCE_THRESHOLD = 0.6

session = requests.Session()


def route(query):
    """Ask the decision model for the target vertical. Returns (answer, latency_ms)."""
    payload = {
        "model": MODEL,
        "state": query,
        "questions": {
            "vertical": {
                "type": "choice",
                "instructions": "Zu welchem Check24 Vergleichsbereich soll der Nutzer weitergeleitet werden?",
                "criteria": VERTICALS,
            }
        },
    }
    start = time.perf_counter()
    response = session.post(URL, json=payload)
    latency_ms = (time.perf_counter() - start) * 1000
    return response.json()["answers"]["vertical"], latency_ms


def redirect_url(answer):
    """Return the redirect URL, or None if the model is unsure or picked 'none'."""
    if answer["choice"] == "none" or answer["confidence"] < CONFIDENCE_THRESHOLD:
        return None
    return REDIRECT_URLS[answer["choice"]]


In [6]:
# Single query playground: change QUERY and re-run
QUERY = "Ich suche eine guenstige Kfz-Versicherung fuer meinen Golf"

answer, latency_ms = route(QUERY)

print(f"query:      {QUERY}")
print(f"vertical:   {answer['choice']}")
print(f"confidence: {answer['confidence']:.2f}")
print(f"redirect:   {redirect_url(answer)}")
print(f"latency:    {latency_ms:.0f} ms")
print("\ntop 3:")
for key, prob in sorted(answer["probabilities"].items(), key=lambda item: -item[1])[:3]:
    print(f"  {key:<18}{prob:.3f}")


query:      Ich suche eine guenstige Kfz-Versicherung fuer meinen Golf
vertical:   kfz_versicherung
confidence: 1.00
redirect:   https://www.check24.de/kfz-versicherung/
latency:    1920 ms

top 3:
  kfz_versicherung  1.000
  konto_karte       0.000
  mietwagen         0.000


In [7]:
# Routing accuracy on a small labelled set: (query, expected vertical)
TEST_QUERIES = [
    ("Mein Stromanbieter hat die Preise erhoeht, ich will wechseln", "strom_gas"),
    ("gas tarif vergleich 20000 kwh", "strom_gas"),
    ("Kfz Versicherung wechseln bis 30.11.", "kfz_versicherung"),
    ("vollkasko tesla model 3", "kfz_versicherung"),
    ("Glasfaser 1000 Mbit verfuegbar in Muenchen?", "internet_dsl"),
    ("handyvertrag mit iphone 17 unbegrenzt daten", "handy"),
    ("Ich brauche 15000 Euro fuer ein neues Auto", "kredit"),
    ("kostenloses girokonto ohne mindesteingang", "konto_karte"),
    ("private haftpflicht familie", "versicherungen"),
    ("malle 2 wochen all inclusive august", "reise"),
    ("Flug Muenchen New York Dezember", "flug"),
    ("mietwagen mallorca flughafen 1 woche", "mietwagen"),
    ("waschmaschine 8kg bosch", "shopping"),
    ("Auto leihen in Lissabon", "mietwagen"),
    ("cheap mobile plan with lots of data", "handy"),
    ("Wo finde ich meine Rechnung?", "none"),
    ("hallo", "none"),
    ("auto", "none"),
]

correct = 0
print(f"{'ok':<4}{'expected':<18}{'predicted':<18}{'conf':>6}{'ms':>7}  {'redirect':<9}query")
for query, expected in TEST_QUERIES:
    answer, latency_ms = route(query)
    hit = answer["choice"] == expected
    correct += hit
    redirect = "yes" if redirect_url(answer) else "no"
    print(f"{'ok' if hit else 'XX':<4}{expected:<18}{answer['choice']:<18}{answer['confidence']:>6.2f}{latency_ms:>7.0f}  {redirect:<9}{query}")

print(f"\naccuracy: {correct}/{len(TEST_QUERIES)} = {correct / len(TEST_QUERIES):.0%}")


ok  expected          predicted           conf     ms  redirect query
ok  strom_gas         strom_gas           0.88    869  yes      Mein Stromanbieter hat die Preise erhoeht, ich will wechseln
ok  strom_gas         strom_gas           0.99    856  yes      gas tarif vergleich 20000 kwh
ok  kfz_versicherung  kfz_versicherung    1.00    862  yes      Kfz Versicherung wechseln bis 30.11.
ok  kfz_versicherung  kfz_versicherung    0.60    852  no       vollkasko tesla model 3
ok  internet_dsl      internet_dsl        0.88    917  yes      Glasfaser 1000 Mbit verfuegbar in Muenchen?
ok  handy             handy               0.95    868  yes      handyvertrag mit iphone 17 unbegrenzt daten
XX  kredit            shopping            0.41    875  no       Ich brauche 15000 Euro fuer ein neues Auto
ok  konto_karte       konto_karte         0.88    850  yes      kostenloses girokonto ohne mindesteingang
ok  versicherungen    versicherungen      0.72    841  yes      private haftpflicht familie
X

In [8]:
# Latency benchmark. The server caches prompts it has already seen, so repeated queries
# are much faster than new ones. "new" appends a unique suffix to force a never-seen query.
N_RUNS = 50


def summary(latencies):
    """Format mean, median, p95, min and max of latencies in ms."""
    p95 = statistics.quantiles(latencies, n=20)[18]
    return f"{statistics.mean(latencies):>7.0f}{statistics.median(latencies):>8.0f}{p95:>7.0f}{min(latencies):>7.0f}{max(latencies):>7.0f}"


queries = [TEST_QUERIES[i % len(TEST_QUERIES)][0] for i in range(N_RUNS)]

route(queries[0])
new = [route(f"{query} {time.time_ns()}")[1] for query in queries]
repeated = [route(query)[1] for query in queries]

print(f"{'queries':<10}{'mean':>7}{'median':>8}{'p95':>7}{'min':>7}{'max':>7}   (ms, {N_RUNS} runs)")
print(f"{'new':<10}{summary(new)}")
print(f"{'repeated':<10}{summary(repeated)}")

queries      mean  median    p95    min    max   (ms, 50 runs)
new          1112    1246   1287    883   1322
repeated       52      51     60     45     65
